# Cosmic Black Swan

TESS light-curve screening and candidate review.

**Workflow:** retrieve TESS products, extract light-curve features, rank observations, then run selected analysis stages.

**Setup:** attach the project ZIP as a Kaggle input and enable Internet. Add the Drive OAuth secret before using checkpoint or restore cells. Select a GPU for Transformer training.


In [ ]:
import pandas as pd

In [ ]:
from pathlib import Path
import zipfile

input_root = Path('/kaggle/input')
working_root = Path('/kaggle/working')
project_roots = [p.parent for p in input_root.rglob('pyproject.toml') if (p.parent / 'src' / 'cosmic_black_swan').is_dir()]
if not project_roots:
    archives = list(input_root.rglob('*.zip'))
    if len(archives) != 1:
        raise FileNotFoundError(f'Expected one project ZIP under /kaggle/input; found: {archives}')
    project_home = working_root / 'cbs-project'
    project_home.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or '..' in member_path.parts:
                raise ValueError(f'Unsafe path in project ZIP: {member.filename}')
        archive.extractall(project_home)
    project_roots = [p.parent for p in project_home.rglob('pyproject.toml') if (p.parent / 'src' / 'cosmic_black_swan').is_dir()]
if len(project_roots) != 1:
    raise RuntimeError(f'Expected exactly one Cosmic Black Swan project root; found {project_roots}')
PROJECT = project_roots[0]
DATA_ROOT = working_root / 'cbs-data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT)
print('Data:', DATA_ROOT)

In [ ]:
import shutil

source_project = PROJECT
PROJECT = Path('/kaggle/working/cbs-project-src')
if source_project.resolve() != PROJECT.resolve():
    shutil.copytree(source_project, PROJECT, dirs_exist_ok=True)

print('Project copy:', PROJECT)

In [ ]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--verbose", "-e", str(PROJECT)],
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print(result.stdout[-6000:])
print(result.stderr[-6000:])

## Resume from Google Drive

Add `GOOGLE_DRIVE_OAUTH_TOKEN_JSON` under Kaggle Secrets. Enter a saved snapshot ID in the next cell to restore a previous run; leave it blank to start fresh.


In [ ]:
import os
import subprocess

def run(args):
    args = [str(x) for x in args]
    print('$', ' '.join(args))
    subprocess.run(args, check=True)

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for secret_name in ['GOOGLE_DRIVE_OAUTH_TOKEN_JSON', 'GEMINI_API_KEY', 'GOOGLE_DRIVE_FOLDER_ID']:
        try:
            os.environ[secret_name] = secrets.get_secret(secret_name)
            print(f'Loaded Kaggle Secret: {secret_name}')
        except Exception:
            continue
except Exception as exc:
    print('Kaggle Secrets are unavailable; pipeline work can continue locally in this session.', type(exc).__name__)

RESTORE_SNAPSHOT_ID = ''
if RESTORE_SNAPSHOT_ID:
    run(['cbs', 'drive-restore', '--snapshot-id', RESTORE_SNAPSHOT_ID, '--destination', DATA_ROOT])

In [ ]:
print("Manifest:", (DATA_ROOT / "manifest.csv").exists())
print("Features:", (DATA_ROOT / "features.csv").exists())
print("Series:", (DATA_ROOT / "series.npy").exists())
print("Raw FITS files:", len(list((DATA_ROOT / "raw").rglob("*.fits"))))

In [ ]:
saved_manifests = list(Path("/kaggle/input").rglob("manifest.csv"))
print(saved_manifests)

In [ ]:
saved_data = Path("/kaggle/input/notebooks/urajkaushik/notebookd2416faa3b/cbs-data")
shutil.copytree(saved_data, DATA_ROOT, dirs_exist_ok=True)

print("Manifest rows:", sum(1 for _ in (DATA_ROOT / "manifest.csv").open()) - 1)
print("Features:", (DATA_ROOT / "features.csv").exists())
print("Series:", (DATA_ROOT / "series.npy").exists())
print("Raw FITS files:", len(list((DATA_ROOT / "raw").rglob("*.fits"))))

In [ ]:
run(["cbs", "featurize", "--data-root", DATA_ROOT])

In [ ]:
run(["cbs", "score", "--data-root", DATA_ROOT, "--contamination", "0.02"])

In [ ]:
run(["cbs", "injection-recovery", "--data-root", DATA_ROOT, "--sample-size", "2000", "--seed", "42"])

In [ ]:
injections = pd.read_csv(DATA_ROOT / "scores" / "injection_recovery_rows.csv")

recovery_summary = (
    injections.groupby(["injection_family", "amplitude_sigma"])
    .agg(
        injections=("recovered_at_frozen_1pct_threshold", "size"),
        recovered=("recovered_at_frozen_1pct_threshold", "sum"),
        recall=("recovered_at_frozen_1pct_threshold", "mean"),
        median_score=("anomaly_score", "median"),
        p90_score=("anomaly_score", lambda scores: scores.quantile(0.90)),
    )
    .reset_index()
)

display(recovery_summary)

## 1. Pilot

Fetch up to 300 light curves from sectors 1-10, extract features, and rank them with Isolation Forest.


In [ ]:
run(['cbs', 'fetch', '--data-root', DATA_ROOT, '--sectors', '1-10', '--max-lightcurves', '300', '--per-sector-cap', '100'])
run(['cbs', 'featurize', '--data-root', DATA_ROOT, '--append'])
run(['cbs', 'score', '--data-root', DATA_ROOT, '--contamination', '0.02'])

In [ ]:
import pandas as pd

manifest = pd.read_csv(DATA_ROOT / 'manifest.csv', dtype={'source_id': 'string'})
complete = manifest[manifest['status'].astype(str).isin(['COMPLETE', 'ARCHIVED'])]
features = pd.read_csv(DATA_ROOT / 'features.csv', dtype={'source_id': 'string'})
print('Downloaded or Drive-archived sector-level products:', len(complete))
manifest_tics = complete['source_id'].astype('string')
print('Unique TICs among completed manifest rows:', manifest_tics[manifest_tics.str.fullmatch(r'\d{8,16}', na=False)].nunique())
print('Unique TICs among usable feature rows:', features['source_id'].astype('string')[features['source_id'].astype('string').str.fullmatch(r'\d{8,16}', na=False)].nunique())
raw_bytes = sum(p.stat().st_size for p in (DATA_ROOT / 'raw').rglob('*.fits'))
print(f'Current raw FITS size: {raw_bytes / (1024**3):.2f} GiB')
print('Usable feature rows:', len(features))
scores = pd.read_csv(DATA_ROOT / 'scores' / 'isolation_forest.csv')
display(scores.head(20))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

ranked = scores.sort_values("candidate_score_v1", ascending=False)

top = ranked.head(5).copy()
top["review_group"] = "Top-ranked"
lower = ranked.tail(5).copy()
lower["review_group"] = "Lower-ranked"

review = pd.concat([top, lower]).drop_duplicates("observation_id")
fig, axes = plt.subplots(len(review), 1, figsize=(12, 18))
axes = np.atleast_1d(axes)

for ax, (_, row) in zip(axes, review.iterrows()):
    with fits.open(row["path"], memmap=False) as hdul:
        data = hdul[1].data
        columns = set(data.dtype.names)
        flux_column = "PDCSAP_FLUX" if "PDCSAP_FLUX" in columns else "SAP_FLUX"
        time = np.array(data["TIME"], dtype=float, copy=True)
        flux = np.array(data[flux_column], dtype=float, copy=True)
        quality = np.array(data["QUALITY"], copy=True)

    valid = np.isfinite(time) & np.isfinite(flux)
    quality_good = valid & (quality == 0)
    if quality_good.sum() >= 32:
        valid = quality_good

    median_flux = np.nanmedian(flux[valid])
    ax.plot(time[valid], flux[valid] / median_flux - 1, ".", markersize=1)
    ax.set_title(
        f'{row["review_group"]}: TIC {row["source_id"]}, '
        f'Sector {row["sector"]}, score {row["candidate_score_v1"]:.3f}'
    )
    ax.set_ylabel("Relative flux")

axes[-1].set_xlabel("Time (BTJD)")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.timeseries import BoxLeastSquares

targets = [(237342298, 3), (402339590, 2)]
durations = np.array([0.05, 0.08, 0.12, 0.20, 0.35, 0.45])

fig, axes = plt.subplots(len(targets), 2, figsize=(14, 8))

for i, (tic, sector) in enumerate(targets):
    match = scores[
        (scores["source_id"].astype(str) == str(tic))
        & (scores["sector"] == sector)
    ].iloc[0]

    with fits.open(match["path"], memmap=False) as hdul:
        data = hdul[1].data
        columns = set(data.dtype.names)
        flux_column = "PDCSAP_FLUX" if "PDCSAP_FLUX" in columns else "SAP_FLUX"
        time = np.array(data["TIME"], dtype=float, copy=True)
        flux = np.array(data[flux_column], dtype=float, copy=True)
        quality = np.array(data["QUALITY"], copy=True)

    valid = np.isfinite(time) & np.isfinite(flux)
    quality_good = valid & (quality == 0)
    if quality_good.sum() >= 32:
        valid = quality_good

    time = time[valid]
    flux = flux[valid]
    flux = flux / np.nanmedian(flux) - 1

    bls = BoxLeastSquares(time, flux)
    result = bls.autopower(
        durations,
        minimum_period=0.5,
        maximum_period=10.0,
        minimum_n_transit=3,
    )

    best = np.argmax(result.power)
    period = result.period[best]
    transit_time = result.transit_time[best]
    phase = ((time - transit_time + period / 2) % period) / period - 0.5

    axes[i, 0].plot(result.period, result.power, ".", markersize=2)
    axes[i, 0].axvline(period, color="red", linestyle="--")
    axes[i, 0].set(title=f"TIC {tic}, sector {sector}: best period {period:.4f} d",
                   xlabel="Period (days)", ylabel="BLS power")

    axes[i, 1].scatter(phase, flux, s=2)
    axes[i, 1].set(xlabel="Phase", ylabel="Relative flux")

    print(f"TIC {tic}, sector {sector}: best trial period = {period:.5f} days")

plt.tight_layout()
plt.show()

In [ ]:
row = scores[
    (scores["source_id"].astype(str) == "402339590")
    & (scores["sector"] == 2)
].iloc[0]

with fits.open(row["path"], memmap=False) as hdul:
    data = hdul[1].data
    columns = set(data.dtype.names)
    flux_column = "PDCSAP_FLUX" if "PDCSAP_FLUX" in columns else "SAP_FLUX"
    time = np.array(data["TIME"], dtype=float, copy=True)
    flux = np.array(data[flux_column], dtype=float, copy=True)
    quality = np.array(data["QUALITY"], copy=True)

valid = np.isfinite(time) & np.isfinite(flux)
quality_good = valid & (quality == 0)
if quality_good.sum() >= 32:
    valid = quality_good

time = time[valid]
flux = flux[valid] / np.nanmedian(flux[valid]) - 1

period = 2 * 3.31064
epoch = time[np.argmin(flux)]
phase = ((time - epoch + period / 2) % period) / period - 0.5

plt.figure(figsize=(9, 4))
plt.scatter(phase, flux, s=2)
plt.xlabel("Phase")
plt.ylabel("Relative flux")
plt.title(f"TIC 402339590 folded at {period:.5f} days")
plt.show()

In [ ]:
tic = "402339590"

tic_rows = features.loc[
    features["source_id"].astype(str) == tic,
    ["source_id", "observation_id", "sector"]
].sort_values("sector")

display(tic_rows)
print(f"Found {len(tic_rows)} observations in the downloaded pilot.")

In [ ]:
from astroquery.mast import Observations

mast_obs = Observations.query_criteria(
    objectname="TIC 402339590",
    obs_collection="TESS",
    dataproduct_type="timeseries",
)

print("TESS observations found:", len(mast_obs))
cols = [
    c for c in ["target_name", "obs_id", "sequence_number", "start_time", "end_time"]
    if c in mast_obs.colnames
]
display(mast_obs[cols] if cols else mast_obs)

In [ ]:
import numpy as np
from astroquery.mast import Observations

tic = "402339590"
sectors_to_check = [69, 96]

tic_obs = mast_obs[
    (mast_obs["target_name"].astype(str) == tic)
    & np.isin(mast_obs["sequence_number"], sectors_to_check)
]

products = Observations.get_product_list(tic_obs)
lc_products = Observations.filter_products(
    products,
    productSubGroupDescription="LC",
    extension="fits",
)

print("Light-curve files found:", len(lc_products))
cols = [
    c for c in [
        "obsID", "productFilename", "productSubGroupDescription",
        "productType", "size"
    ]
    if c in lc_products.colnames
]
display(lc_products[cols])

In [ ]:
from pathlib import Path
from astroquery.mast import Observations

check_dir = Path("/kaggle/working/tic_402339590_check")
check_dir.mkdir(parents=True, exist_ok=True)

download_manifest = Observations.download_products(
    lc_products,
    download_dir=str(check_dir),
    flat=True,
    verbose=True,
)

display(download_manifest)

In [ ]:
from astropy.io import fits
from astropy.timeseries import BoxLeastSquares
import numpy as np
import matplotlib.pyplot as plt

files = sorted(check_dir.glob("*_lc.fits"))
if len(files) != 2:
    raise RuntimeError(f"Expected 2 light curves; found {len(files)}: {files}")

trial_period = 6.62128
durations = np.array([0.03, 0.05, 0.08, 0.12, 0.18])  # days

fig, axes = plt.subplots(len(files), 2, figsize=(13, 8))
if len(files) == 1:
    axes = np.array([axes])

for row, path in enumerate(files):
    with fits.open(path, memmap=False) as hdul:
        data = hdul[1].data
        names = data.names
        flux_name = "PDCSAP_FLUX" if "PDCSAP_FLUX" in names else "SAP_FLUX"

        time = np.asarray(data["TIME"], dtype=float)
        flux = np.asarray(data[flux_name], dtype=float)
        quality = np.asarray(data["QUALITY"])

    valid = np.isfinite(time) & np.isfinite(flux) & (quality == 0)
    if valid.sum() < 32:
        valid = np.isfinite(time) & np.isfinite(flux)

    time, flux = time[valid], flux[valid]
    order = np.argsort(time)
    time, flux = time[order], flux[order]
    relative_flux = flux / np.median(flux) - 1

    span = time.max() - time.min()
    maximum_period = min(10.0, span / 3)
    if maximum_period <= 0.5:
        raise RuntimeError(f"Not enough time coverage in {path.name} for this search.")

    bls = BoxLeastSquares(time, relative_flux)
    result = bls.autopower(
        durations,
        minimum_period=0.5,
        maximum_period=maximum_period,
    )
    best = np.argmax(result.power)
    print(
        f"{path.name}: best BLS period = "
        f"{result.period[best]:.5f} days; {len(time)} usable points"
    )

    axes[row, 0].plot(time, relative_flux, ".", ms=1)
    axes[row, 0].set_title(f"{path.name} — time series")
    axes[row, 0].set_xlabel("Time (BTJD)")
    axes[row, 0].set_ylabel("Relative flux")

    phase = ((time - time[0] + trial_period / 2) % trial_period) / trial_period - 0.5
    axes[row, 1].plot(phase, relative_flux, ".", ms=1)
    axes[row, 1].set_title(f"Folded at {trial_period:.5f} days")
    axes[row, 1].set_xlabel("Phase")
    axes[row, 1].set_ylabel("Relative flux")

plt.tight_layout()
plt.show()

In [ ]:
period_grid = np.linspace(6.0, 7.2, 801)
durations = np.array([0.05, 0.10, 0.20, 0.30, 0.45, 0.60])

for path in files:
    with fits.open(path, memmap=False) as hdul:
        data = hdul[1].data
        flux_name = "PDCSAP_FLUX" if "PDCSAP_FLUX" in data.names else "SAP_FLUX"

        time = np.asarray(data["TIME"], dtype=float)
        flux = np.asarray(data[flux_name], dtype=float)
        quality = np.asarray(data["QUALITY"])

    valid = np.isfinite(time) & np.isfinite(flux) & (quality == 0)
    if valid.sum() < 32:
        valid = np.isfinite(time) & np.isfinite(flux)

    time, flux = time[valid], flux[valid]
    order = np.argsort(time)
    time, flux = time[order], flux[order]
    relative_flux = flux / np.median(flux) - 1

    bls = BoxLeastSquares(time, relative_flux)
    best_power = -np.inf

    for duration in durations:
        result = bls.power(period_grid, duration)
        index = np.argmax(result.power)

        if result.power[index] > best_power:
            best_power = result.power[index]
            best_period = period_grid[index]
            best_duration = duration

    print(
        f"{path.name}: best period = {best_period:.5f} days, "
        f"duration = {best_duration:.2f} days"
    )

In [ ]:
tic_sector2 = scores.loc[
    (scores["source_id"].astype(str) == "402339590")
    & (scores["sector"] == 2),
    ["source_id", "observation_id", "sector", "path"]
]

display(tic_sector2)

In [ ]:
observations = [(2, str(tic_sector2.iloc[0]["path"]))]

for path in files:
    if "s0069" in path.name:
        observations.append((69, str(path)))
    elif "s0096" in path.name:
        observations.append((96, str(path)))

all_times = []
all_fluxes = []
all_sectors = []

for sector, path in observations:
    with fits.open(path, memmap=False) as hdul:
        data = hdul[1].data
        flux_name = "PDCSAP_FLUX" if "PDCSAP_FLUX" in data.names else "SAP_FLUX"

        time = np.asarray(data["TIME"], dtype=float)
        flux = np.asarray(data[flux_name], dtype=float)
        quality = np.asarray(data["QUALITY"])

    valid = np.isfinite(time) & np.isfinite(flux) & (quality == 0)
    if valid.sum() < 32:
        valid = np.isfinite(time) & np.isfinite(flux)

    time, flux = time[valid], flux[valid]
    relative_flux = flux / np.median(flux) - 1

    all_times.append(time)
    all_fluxes.append(relative_flux)
    all_sectors.append(np.full(len(time), sector))

time = np.concatenate(all_times)
relative_flux = np.concatenate(all_fluxes)
sector_labels = np.concatenate(all_sectors)

order = np.argsort(time)
time = time[order]
relative_flux = relative_flux[order]
sector_labels = sector_labels[order]

period_grid = np.linspace(6.55, 6.70, 601)
durations = np.array([0.10, 0.20, 0.40, 0.60])

bls = BoxLeastSquares(time, relative_flux)
best_power = -np.inf

for duration in durations:
    result = bls.power(period_grid, duration)
    index = np.nanargmax(result.power)

    if result.power[index] > best_power:
        best_power = result.power[index]
        best_period = period_grid[index]
        best_duration = duration
        best_epoch = float(result.transit_time[index])

print(
    f"Combined best period: {best_period:.5f} days; "
    f"duration tested: {best_duration:.2f} days"
)

phase = ((time - best_epoch + best_period / 2) % best_period) / best_period - 0.5

for sector in [2, 69, 96]:
    mask = sector_labels == sector
    plt.scatter(
        phase[mask], relative_flux[mask],
        s=2, alpha=0.25, label=f"Sector {sector}"
    )

plt.xlabel("Phase")
plt.ylabel("Relative flux")
plt.title(f"All sectors folded at {best_period:.5f} days")
plt.legend()
plt.show()

In [ ]:
tpf_observations = mast_obs[
    (mast_obs["target_name"].astype(str) == "402339590")
    & np.isin(mast_obs["sequence_number"], [69, 96])
]

tpf_products = Observations.filter_products(
    Observations.get_product_list(tpf_observations),
    productSubGroupDescription="TP",
    extension="fits",
)

print("Target-pixel files found:", len(tpf_products))
cols = [
    c for c in ["obsID", "productFilename", "productSubGroupDescription", "size"]
    if c in tpf_products.colnames
]
display(tpf_products[cols])

In [ ]:
tpf_dir = check_dir / "target_pixels"
tpf_dir.mkdir(parents=True, exist_ok=True)

tpf_downloads = Observations.download_products(
    tpf_products,
    download_dir=str(tpf_dir),
    flat=True,
    verbose=True,
)

display(tpf_downloads)

In [ ]:
tpf_files = sorted(tpf_dir.glob("*_tp.fits"))
if len(tpf_files) != 2:
    raise RuntimeError(f"Expected 2 target-pixel files; found {len(tpf_files)}")

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for row, path in enumerate(tpf_files):
    with fits.open(path, memmap=False) as hdul:
        data = hdul[1].data
        time = np.asarray(data["TIME"], dtype=float)
        cube = np.asarray(data["FLUX"], dtype=float)
        quality = np.asarray(data["QUALITY"])

        aperture_hdu = next(
            (hdu for hdu in hdul if hdu.name.upper() == "APERTURE"),
            None,
        )
        aperture = (
            None if aperture_hdu is None
            else np.asarray(aperture_hdu.data)
        )

    phase = ((time - best_epoch + best_period / 2) % best_period) / best_period - 0.5
    good = np.isfinite(time) & (quality == 0)

    in_eclipse = good & (np.abs(phase) < 0.05)
    out_of_eclipse = good & (np.abs(phase) > 0.14) & (np.abs(phase) < 0.36)

    if in_eclipse.sum() == 0 or out_of_eclipse.sum() == 0:
        raise RuntimeError(f"Not enough eclipse or comparison cadences in {path.name}")

    out_image = np.nanmedian(cube[out_of_eclipse], axis=0)
    in_image = np.nanmedian(cube[in_eclipse], axis=0)
    difference = out_image - in_image

    sector = "Sector 69" if "s0069" in path.name else "Sector 96"
    print(
        f"{sector}: {in_eclipse.sum()} eclipse cadences, "
        f"{out_of_eclipse.sum()} comparison cadences"
    )

    images = [out_image, in_image, difference]
    titles = ["Outside eclipse", "Deep eclipse", "Outside − eclipse"]

    for col, (image_data, title) in enumerate(zip(images, titles)):
        axes[row, col].imshow(image_data, origin="lower", interpolation="nearest")
        axes[row, col].set_title(f"{sector}: {title}")
        axes[row, col].set_xlabel("Pixel column")
        axes[row, col].set_ylabel("Pixel row")

    if aperture is not None and np.any(aperture > 0):
        axes[row, 2].contour(
            aperture > 0, levels=[0.5], colors="white", linewidths=0.8
        )

plt.tight_layout()
plt.show()

In [ ]:
run(["cbs", "--help"])

In [ ]:
run(["cbs", "injection-recovery", "--help"])

In [ ]:
run([
    "cbs", "injection-recovery",
    "--data-root", DATA_ROOT,
    "--sample-size", "100",
    "--seed", "42",
])

In [ ]:
sensitivity_map = pd.read_csv(
    DATA_ROOT / "scores" / "blindspot_sensitivity_map.csv"
)

print("Rows:", len(sensitivity_map))
print("Columns:", sensitivity_map.columns.tolist())
display(sensitivity_map.head(20))

In [ ]:
sensitivity_map["recovered"] = np.rint(
    sensitivity_map["recall"] * sensitivity_map["injections"]
).astype(int)

recovery_summary = (
    sensitivity_map
    .groupby(["injection_family", "amplitude_sigma"], as_index=False)
    .agg(
        injections=("injections", "sum"),
        recovered=("recovered", "sum"),
    )
)

recovery_summary["recall"] = (
    recovery_summary["recovered"] / recovery_summary["injections"]
)

print("Total injections:", recovery_summary["injections"].sum())
display(recovery_summary.sort_values(["injection_family", "amplitude_sigma"]))

In [ ]:
needles = (
    "blindspot_sensitivity_map.csv",
    "frozen_threshold_baseline_quantile",
)

for source_file in PROJECT.rglob("*.py"):
    lines = source_file.read_text(
        encoding="utf-8", errors="ignore"
    ).splitlines()

    for i, line in enumerate(lines):
        if any(needle in line for needle in needles):
            print(source_file)
            start = max(0, i - 15)
            end = min(len(lines), i + 36)
            print("\n".join(
                f"{j + 1}: {lines[j]}" for j in range(start, end)
            ))
            break

In [ ]:
needle = "background_sources_sampled"

for source_file in PROJECT.rglob("*.py"):
    lines = source_file.read_text(
        encoding="utf-8", errors="ignore"
    ).splitlines()

    for i, line in enumerate(lines):
        if needle in line:
            print(source_file)
            start = max(0, i - 30)
            end = min(len(lines), i + 70)
            print("\n".join(
                f"{j + 1}: {lines[j]}" for j in range(start, end)
            ))
            break

In [ ]:
injection_rows = pd.read_csv(
    DATA_ROOT / "scores" / "injection_recovery_rows.csv"
)
threshold = float(sensitivity_map["baseline_threshold_score"].iloc[0])

raw_score_summary = (
    injection_rows
    .groupby(["injection_family", "amplitude_sigma"], as_index=False)
    .agg(
        injections=("anomaly_score", "size"),
        median_score=("anomaly_score", "median"),
        p90_score=("anomaly_score", lambda values: values.quantile(0.90)),
        max_score=("anomaly_score", "max"),
        recovered=("recovered_at_frozen_1pct_threshold", "sum"),
    )
)

raw_score_summary["recall"] = (
    raw_score_summary["recovered"] / raw_score_summary["injections"]
)

print("Frozen threshold:", threshold)
display(raw_score_summary)

In [ ]:
source_file = PROJECT / "src" / "cosmic_black_swan" / "injections.py"
lines = source_file.read_text(encoding="utf-8", errors="ignore").splitlines()

print("\n".join(
    f"{i + 1}: {line}" for i, line in enumerate(lines[:160])
))

In [ ]:
if os.environ.get('GOOGLE_DRIVE_OAUTH_TOKEN_JSON'):
    run(['cbs', 'drive-backup', '--data-root', DATA_ROOT, '--shard-size-mb', '512'])
else:
    print('Drive checkpoint skipped; configure the Drive secret to enable it.')


## 2. Expand the dataset

Set a cumulative target in the next cell. Each batch fetches new products, updates features and scores, archives raw files, and saves a resumable snapshot.


In [ ]:
run(["cbs", "fetch", "--data-root", DATA_ROOT,
     "--sectors", "1-10", "--max-lightcurves", "5000",
     "--per-sector-cap", "500"])

In [ ]:
run(["cbs", "featurize", "--data-root", DATA_ROOT, "--append"])

In [ ]:
run(["cbs", "score", "--data-root", DATA_ROOT, "--contamination", "0.02"])

In [ ]:
run(["cbs", "injection-recovery", "--data-root", DATA_ROOT,
     "--sample-size", "2000", "--seed", "42"])

In [ ]:
ir = pd.read_csv(DATA_ROOT / "scores" / "injection_recovery_rows.csv")

summary = (
    ir.groupby(["injection_family", "amplitude_sigma"])
      .agg(
          injections=("recovered_at_frozen_1pct_threshold", "size"),
          recovered=("recovered_at_frozen_1pct_threshold", "sum"),
          recall=("recovered_at_frozen_1pct_threshold", "mean"),
          median_score=("anomaly_score", "median"),
          p90_score=("anomaly_score", lambda s: s.quantile(0.90)),
      )
      .reset_index()
)

display(summary)

In [ ]:
from cosmic_black_swan.cloud_drive import backup_to_google_drive
from cosmic_black_swan.storage import prune_archived_lightcurves

CUMULATIVE_TARGET = 2000
run(['cbs', 'fetch', '--data-root', DATA_ROOT, '--sectors', '1-96', '--max-lightcurves', CUMULATIVE_TARGET])
run(['cbs', 'featurize', '--data-root', DATA_ROOT, '--append'])
run(['cbs', 'score', '--data-root', DATA_ROOT])
if not os.environ.get('GOOGLE_DRIVE_OAUTH_TOKEN_JSON'):
    raise RuntimeError('Set GOOGLE_DRIVE_OAUTH_TOKEN_JSON in Kaggle Secrets before large-batch pruning.')
source_snapshot = backup_to_google_drive(DATA_ROOT)
prune_result = prune_archived_lightcurves(DATA_ROOT, source_snapshot['snapshot_id'])
resumable_snapshot = backup_to_google_drive(DATA_ROOT)
print('Source FITS archive snapshot:', source_snapshot['snapshot_id'])
print('Local raw files pruned:', prune_result['raw_lightcurve_files_removed'])
print('Raw files retained for review:', len(prune_result['raw_files_retained']))
print('Restore this post-prune snapshot next session:', resumable_snapshot['snapshot_id'])

## 3. Model and candidate analysis

Run selected analysis stages after the data pipeline is stable. See the README for XGBoost, Transformer, conformal screening, physics inference, pixel audits, observation planning, and candidate dossiers.
